In [1]:
import pandas as pd
import numpy as np

df_near = pd.read_csv("../data/near_earth_asteroids_2025.csv")

df_near.rename({'q':'perihelion_distance_au', 'ad':'aphelion_distance_au', 'e':'eccentricity', 'a':'semimajor_axis_au'\
               , 'i':'inclination_deg','per':'orbital_period_days','n':'mean_motion_deg_day', 'rot_per':'rot_per_h',\
                'class':'class_code'}, axis=1, inplace=True)

df_close = pd.read_csv("../data/asteroid_close_approaches_2015_2035.csv")

df_close['close_approach_date'] = pd.to_datetime(df_close['close_approach_date'], format="ISO8601")

df_near.describe()


/tmp/ipykernel_59930/4288578708.py:4: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df_near = pd.read_csv("../data/near_earth_asteroids_2025.csv")


,spkid,H,diameter_km,diameter_m,albedo,rot_per_h,eccentricity,semimajor_axis_au,inclination_deg,perihelion_distance_au,aphelion_distance_au,orbital_period_days,per_y,moid_au,moid_km,moid_lunar_distances,mean_motion_deg_day,condition_code,data_arc,data_arc_years
count,4.128100e+04,41279.000000,41281.000000,41281.000000,1204.000000,2181.000000,41281.000000,41281.000000,41281.000000,41281.000000,41281.000000,4.128100e+04,41281.000000,4.115000e+04,4.115000e+04,41150.000000,41281.000000,41279.000000,40875.000000,40875.000000
mean,2.852738e+07,23.718323,0.165592,165.588917,0.172680,17.249923,0.434323,1.757387,11.834802,0.914116,2.600654,9.706043e+02,2.657599,8.217933e-02,1.229385e+07,31.981713,0.530476,5.262506,1404.433394,3.845177
std,2.428187e+07,2.876761,0.403793,403.794103,0.144368,71.023057,0.176975,2.119936,10.549392,0.221873,4.221280,1.262065e+04,34.619953,9.638212e-02,1.441856e+07,37.509020,0.284677,3.169935,3099.025028,8.484637
min,3.002856e+06,9.170000,0.000475,0.500000,0.009000,0.000832,0.002800,0.461800,0.010000,0.069000,0.650000,1.150000e+02,0.314000,4.540000e-07,6.800000e+01,0.000000,0.000150,0.000000,0.000000,0.000000
25%,3.745029e+06,21.675000,0.024573,24.600000,0.047000,1.300000,0.300300,1.287000,4.350000,0.792000,1.650000,5.330000e+02,1.460000,1.160000e-02,1.735335e+06,4.510000,0.309600,2.000000,6.000000,0.020000
50%,2.046734e+07,24.040000,0.055266,55.300000,0.137000,4.400000,0.448200,1.679000,8.400000,0.962000,2.410000,7.950000e+02,2.180000,4.230000e-02,6.327990e+06,16.460000,0.453000,7.000000,22.000000,0.060000
75%,5.433661e+07,25.800000,0.165372,165.400000,0.256000,10.580000,0.563000,2.164000,16.550000,1.057000,3.340000,1.160000e+03,3.180000,1.210000e-01,1.810134e+07,47.090000,0.675200,8.000000,855.500000,2.345000
max,5.460683e+07,34.370000,37.675000,37675.000000,0.856000,1880.000000,0.996400,350.300000,165.600000,1.300000,699.320000,2.390000e+06,6560.000000,7.080000e-01,1.059153e+08,275.530000,3.141000,9.000000,46582.000000,127.530000


In [2]:

df_near[df_near['last_obs'] < "2015-01-01"] ## all asteroids that were last seen before 2015. We can leave these out


df_close[df_close['designation'] == "2022 AP1"]
df_near['full_name'] = df_near['full_name'].apply(lambda x: x[x.index('('): ])
df_close['full_name'] = df_close['full_name'].apply(lambda x: x[x.index('('):] if '(' in x else np.nan)
df_close = df_close.dropna(how='any', subset=['full_name'], axis=0)
df_close['full_name'].isna().sum()

df_close.reset_index(drop=True, inplace=True)





In [3]:
## Now we can finally join df_near and df_close on the column 'full_name'

df_near['full_name'] = df_near['full_name'].str.strip()
df_close['full_name'] = df_close['full_name'].str.strip()


In [4]:
import datetime
X_near = df_near.drop(['spkid','full_name','pdes','name', 'diameter_m','moid_lunar_distances','albedo', 'rot_per_h',\
                       'moid_km', 'per_y', 'data_arc_years', 'diameter_is_estimated','pha', 'first_obs', 'last_obs'], axis=1)
# per --> how many Earth days the asteroid takes to complete one full revolution
# per_y --> how many Earth years the asteroid takes to complete one full revolution
X_near['size_category'] = X_near['size_category'].apply(lambda x: x[0:x.index(" ")].strip())
y = df_near['pha']


X_near_no_moid = df_near.drop(['spkid','full_name','pdes','name', 'diameter_m','moid_lunar_distances','albedo','rot_per_h',\
                       'moid_km','moid_au', 'per_y', 'data_arc_years', 'diameter_is_estimated', 'first_obs', 'last_obs'], axis=1)
# the dataframe X_near_no_moid is the dataset we will use to fit a model to predict moid.
X_near_no_moid['size_category'] = X_near_no_moid['size_category'].apply(lambda x: x[0:x.index(" ")].strip())
y_moid = df_near['moid_au']

X_near_to_merge = df_near.drop(['spkid','pdes','name', 'diameter_m','moid_lunar_distances','albedo', 'rot_per_h',\
                       'moid_km', 'per_y', 'data_arc_years', 'diameter_is_estimated', 'first_obs', 'last_obs', 'H'], axis=1)
X_near_to_merge['size_category'] = X_near_to_merge['size_category'].apply(lambda x: x[0:x.index(" ")].strip())



#X_near.info()
X_near['moid_au'].describe()
X_close = df_close.copy()

X_close['is_future'] = np.where(X_close['close_approach_date'] >= datetime.datetime.now(), True, False)
X_close['is_future'].value_counts() # there are 44 less True values in column 'is_future'

X_merged = pd.merge(X_near_to_merge, X_close, on='full_name', how='inner').drop(['dist_km', 'dist_lunar'], axis=1)
y_absolute_magnitude = X_merged['absolute_magnitude']

X_merged.drop(["full_name", "designation", "close_approach_date","velocity_km_s", "velocity_infinity_km_s",\
               "distance_min_au","distance_max_au","absolute_magnitude"], axis=1, inplace=True)
# the dataset X_merged above will be used to predict 'absolute_magnitude'
X_merged






,pha,diameter_km,size_category,class_code,eccentricity,semimajor_axis_au,inclination_deg,perihelion_distance_au,aphelion_distance_au,orbital_period_days,moid_au,mean_motion_deg_day,condition_code,data_arc,distance_au,v_rel_kmh,is_future
0,False,4.200000,Large,AMO,0.5712,2.4740,9.40,1.061,3.89,1420.0,0.079700,0.2533,0.0,39281.0,0.082198,29695.0,False
1,True,1.000000,Large,APO,0.8270,1.0780,22.80,0.186,1.97,409.0,0.033500,0.8805,0.0,27807.0,0.053836,108801.0,False
2,False,5.700000,Large,AMO,0.5055,2.1490,23.96,1.063,3.24,1150.0,0.071700,0.3128,0.0,26251.0,0.089665,52010.0,True
3,True,3.400000,Large,APO,0.6505,1.7760,39.82,0.621,2.93,865.0,0.002770,0.4163,0.0,19295.0,0.089572,95428.0,False
4,True,3.400000,Large,APO,0.6505,1.7760,39.82,0.621,2.93,865.0,0.002770,0.4163,0.0,19295.0,0.086350,95105.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27417,False,0.009873,Tiny,APO,0.2638,1.3000,4.32,0.957,1.64,541.0,0.018100,0.6651,8.0,2.0,0.020381,21554.0,False
27418,False,0.006958,Tiny,ATE,0.3022,0.9757,11.05,0.681,1.27,352.0,0.009540,1.0230,6.0,1.0,0.011928,39123.0,False
27419,False,0.010976,Tiny,APO,0.5668,2.1170,1.53,0.917,3.32,1130.0,0.000852,0.3199,9.0,3.0,0.027740,37764.0,False
27420,False,0.013692,Tiny,APO,0.7362,2.6870,1.66,0.709,4.66,1610.0,0.008870,0.2238,9.0,2.0,0.012432,69399.0,False


In [5]:

pha_counts = df_near['pha'].value_counts()
pha_counts.get(True)/len(df_near)*100

np.float64(6.150529299193334)

In [6]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
import numpy as np



categorical_transformer = Pipeline(
    steps = [
        ("one_hot_encoder", OneHotEncoder(handle_unknown='ignore', drop='first'))
    ]
)

numeric_transformer = Pipeline(
    steps = [
        ('simple_imputer', SimpleImputer(missing_values=np.nan, strategy='mean')),
        ('standard_scaler', StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('categorical_transformer', categorical_transformer, make_column_selector(dtype_include=['object','category','str'])),
        ('numeric_transformer', numeric_transformer, make_column_selector(dtype_include='number'))
    ]
)


pipeline_lr = make_pipeline(preprocessor, LogisticRegression(class_weight='balanced'))
pipeline_rfc = make_pipeline(preprocessor, RandomForestClassifier(class_weight='balanced'))
pipeline_gbc = make_pipeline(preprocessor, GradientBoostingClassifier())

pipeline_dtr = make_pipeline(preprocessor, DecisionTreeRegressor())
pipeline_rfr = make_pipeline(preprocessor, RandomForestRegressor())
pipeline_gbr = make_pipeline(preprocessor, GradientBoostingRegressor())
pipeline_linreg = make_pipeline(preprocessor, LinearRegression())



pipelines_classification = {
    "Random Forest Classifier": pipeline_rfc,
    "Logistic Regression": pipeline_lr,
    "Gradient Boosting Classifier" : pipeline_gbc
}

pipelines_regression = {
    "Random Forest Regressor": pipeline_rfr,
    "Decision Tree Regressor": pipeline_dtr,
    "Gradient Boosting Regressor": pipeline_gbr,
    "Linear Regression": pipeline_linreg
}


X_train, X_test, y_train, y_test = train_test_split(X_near, y, test_size=0.2, random_state=42)
X_train_no_moid, X_test_no_moid, y_train_moid, y_test_moid = train_test_split(X_near_no_moid, y_moid, test_size=0.2, random_state=42)
X_train_merged, X_test_merged, y_train_magnitude, y_test_magnitude = train_test_split(X_merged, y_absolute_magnitude, test_size=0.2, random_state=42)

trainsets = {"X_near": (X_train, y_train, "cat"), 
             
             "X_near_no_moid": (X_train_no_moid, y_train_moid, "num"),

             "X_merged":(X_train_merged, y_train_magnitude,"num")
             }

stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

def cross_val_all(trainsets, pipelines_c, pipelines_r):
    for name, (X_train, y_train, type) in trainsets.items():
        print(f"{name} train data cross-validation:\n")
        if name == "X_merged":
            break
        if type == "cat":

            for name, pipeline in pipelines_c.items():

                if y_train.isna().sum() > 0:

                    y_nan_idx = y_train[y_train.isna()].index.tolist()
                    y_train = y_train.drop(y_nan_idx, axis=0).copy()
                    X_train = X_train.drop(y_nan_idx, axis=0).copy()
                    scores = cross_val_score(pipeline, X_train, y_train, cv=stratified_kf, scoring='roc_auc')
                    results[name] = {
                        'model': pipeline, 'cv_auc': scores.mean(),
                        'cv_auc_std': scores.std()
                    }
                    continue

                scores = cross_val_score(pipeline, X_train, y_train, cv=stratified_kf, scoring='roc_auc')
                results[name] = {
                        'model': pipeline, 'cv_auc': scores.mean(),
                        'cv_auc_std': scores.std()
                    }

                print(f"{name:20s}: ROC-AUC score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")
        else:
            
            for name, pipeline in pipelines_r.items():

                if y_train.isna().sum() > 0:
                    y_nan_idx = y_train[y_train.isna()].index.tolist()
                    y_train = y_train.drop(y_nan_idx, axis=0).copy()
                    X_train = X_train.drop(y_nan_idx, axis=0).copy()
                    scores = cross_val_score(pipeline, X_train, y_train, cv=5)
                    print(f"{name:20s}: r2 score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")
                    continue
                scores = cross_val_score(pipeline, X_train, y_train, cv=5)
                print(f"{name:20s}: r2 score = {np.mean(scores):.3f} ± {np.std(scores):.3f}")

        print("====================")


cross_val_all(trainsets, pipelines_classification, pipelines_regression)




X_near train data cross-validation:

Random Forest Classifier: ROC-AUC score = 0.999 ± 0.001
Logistic Regression : ROC-AUC score = 0.998 ± 0.001
Gradient Boosting Classifier: ROC-AUC score = 0.999 ± 0.001
X_near_no_moid train data cross-validation:

Random Forest Regressor: r2 score = 0.812 ± 0.008
Decision Tree Regressor: r2 score = 0.633 ± 0.015
Gradient Boosting Regressor: r2 score = 0.799 ± 0.005
Linear Regression   : r2 score = 0.504 ± 0.166
X_merged train data cross-validation:



In [7]:
from pathlib import Path

DATA_PATH = Path("../artifacts/data")
DATA_PATH.mkdir(parents=True, exist_ok=True)

numeric_columns = X_train.select_dtypes(include='number')
cat_columns = X_train.select_dtypes(include=['category', 'str', 'object'])
quantiles_x_train = X_train.quantile([.1, .25, .5, .75, .9], axis=0, numeric_only=True)
X_train.describe()

lst = numeric_columns.std().index
vals = numeric_columns.std().values.tolist()

X_train_stats = pd.concat([pd.DataFrame([vals], columns=lst, index=['std']),quantiles_x_train,\
            pd.DataFrame([numeric_columns.min().values.tolist()], columns=lst, index=['min']),\
                         pd.DataFrame([numeric_columns.max().values.tolist()], columns=lst, index=['max'])])
#dftemp = pd.DataFrame(np.array(numeric_columns.std().values).reshape(-1), columns=lst)
X_train_stats.round(3).to_csv(DATA_PATH / "X_train_stats.csv")

#pd.concat([quantiles_x_train, numeric_columns.std()], ignore_index=True, axis=1)


In [8]:
thresholds = np.arange(0.1, 1, 0.1)

best_model = pipelines_classification['Random Forest Classifier']
best_model.fit(X_train, y_train)

y_train_pred = best_model.predict(X_train)
y_train_proba = best_model.predict_proba(X_train)[:, 1]
print(y_train_pred)
y_train_proba > 0

[False False False ... False False  True]


array([False, False, False, ..., False, False,  True], shape=(33024,))

In [9]:
from sklearn.metrics import f1_score
y_train_pred = best_model.predict(X_train)
y_train_proba = best_model.predict_proba(X_train)[:, 1]
print(y_train_pred)

best_f1 = 0
print(f"Initial f1_score: {f1_score(y_train, y_train_pred)}")

for thresh in thresholds:
    hold = [1 if thresh > y_train_proba[i] else 0 for i in range(len(y_train_proba))]
    temp_f1 = f1_score(y_train, hold)
    print(temp_f1)
    if temp_f1 > best_f1:
        best_f1 = temp_f1

nan_indices = []

for col in X_train.columns.tolist():
    if X_train[col].isna().sum() > 0:
        nan_indices.extend(X_train[X_train[col].isna()].index.tolist())
len(nan_indices)

# X_train.isna().sum()

[False False False ... False False  True]
Initial f1_score: 1.0
0.0
0.0
0.0
0.0
0.0
6.056018168054504e-05
0.0009080177971488242
0.0016942998910807214
0.0036874716638960253


433

In [10]:

from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score, classification_report
import joblib

# Best model for PHA

best_model = pipelines_classification["Random Forest Classifier"]

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]
print(y_proba)

print(y_test.sum()/len(y_test))
print(y_pred.sum()/len(y_pred))
print(classification_report(y_test, y_pred))
print(roc_auc_score(y_pred, y_proba))

best_model_columntransformer = best_model['columntransformer']
best_model_columntransformer.fit(X_train)

joblib.dump(best_model['randomforestclassifier'], '../artifacts/best_model.pkl')
joblib.dump(best_model_columntransformer, '../artifacts/best_model_columntransformer.pkl')

#Saving reference train sets for monitoring
joblib.dump(X_train, '../artifacts/data/X_train.pkl')
joblib.dump(y_train, '../artifacts/data/y_train.pkl')

#Saving test sets for simulation of new data during monitoring
joblib.dump(X_test, '../artifacts/data/X_test.pkl')
joblib.dump(y_test, '../artifacts/data/y_test.pkl')


[0. 0. 0. ... 0. 0. 0.]
0.0634613055589197
0.06406685236768803
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      7733
        True       0.98      0.99      0.99       524

    accuracy                           1.00      8257
   macro avg       0.99      1.00      0.99      8257
weighted avg       1.00      1.00      1.00      8257

1.0


['../artifacts/data/y_test.pkl']

In [11]:
from sklearn.metrics import r2_score

best_model_abs_mag = pipelines_regression["Random Forest Regressor"]

best_model_abs_mag.fit(X_train_merged, y_train_magnitude)


#y_pred_abs_mag = best_model_abs_mag.predict(X_test_merged)

#print(f"r2 score: {r2_score(y_pred_abs_mag, y_test_magnitude)}")


joblib.dump(best_model_abs_mag['randomforestregressor'], '../artifacts/best_model_abs_mag.pkl')
column_transformer_abs_mag = best_model_abs_mag['columntransformer']
column_transformer_abs_mag.fit(X_train_merged)


joblib.dump(column_transformer_abs_mag, '../artifacts/column_transformer_abs_mag.pkl')

#Saving reference train sets for monitoring
joblib.dump(X_train_merged, '../artifacts/data/X_train_merged.pkl')
joblib.dump(y_train_magnitude, '../artifacts/data/y_train_magnitude.pkl')

#Saving test sets for simulation of new data during monitoring
joblib.dump(X_test_merged, '../artifacts/data/X_test_merged.pkl')
joblib.dump(y_test_magnitude, '../artifacts/data/y_test_magnitude.pkl')



['../artifacts/data/y_test_magnitude.pkl']

In [12]:
best_model_abs_mag.predict(X_test_merged)

array([24.3   , 30.1   , 22.9   , ..., 29.5601, 24.88  , 25.18  ],
      shape=(5485,))

In [13]:
best_model_abs_mag['columntransformer'].get_feature_names_out()
#best_model['columntransformer'].get_feature_names_out()
X_train_merged


,pha,diameter_km,size_category,class_code,eccentricity,semimajor_axis_au,inclination_deg,perihelion_distance_au,aphelion_distance_au,orbital_period_days,moid_au,mean_motion_deg_day,condition_code,data_arc,distance_au,v_rel_kmh,is_future
16092,False,0.038946,Small,APO,0.4382,1.5560,14.35,0.874,2.24,709.0,0.044700,0.5077,8.0,6.0,0.051563,42880.0,False
19703,False,0.015222,Tiny,APO,0.6472,1.9860,4.49,0.701,3.27,1020.0,0.002050,0.3522,8.0,2.0,0.002731,65874.0,False
5883,False,0.040594,Small,APO,0.2465,1.1670,4.80,0.879,1.45,460.0,0.010500,0.7822,8.0,7.0,0.054310,24931.0,False
1574,False,0.049030,Small,ATE,0.2047,0.8465,13.45,0.673,1.02,284.0,0.027500,1.2650,8.0,2.0,0.063267,30150.0,False
1349,False,0.106773,Small,APO,0.7683,2.4850,3.93,0.576,4.39,1430.0,0.065000,0.2517,2.0,7111.0,0.065928,81672.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21575,False,0.001572,Tiny,APO,0.5625,1.0480,0.52,0.458,1.64,392.0,0.000042,0.9191,7.0,NaN,0.077480,56862.0,False
5390,False,0.029817,Small,ATE,0.3482,0.9391,3.50,0.612,1.27,332.0,0.016400,1.0830,6.0,14.0,0.093279,29915.0,True
860,True,0.319493,Medium,APO,0.5756,1.6060,11.83,0.681,2.53,743.0,0.043000,0.4844,0.0,3643.0,0.072501,69454.0,False
15795,False,0.027068,Small,AMO,0.2351,1.3900,10.50,1.063,1.72,599.0,0.074900,0.6013,7.0,28.0,0.077275,22105.0,False


In [14]:


moid_gbr_param_grid = {
    "gradientboostingregressor__n_estimators": [10, 30, 100, 300],
    "gradientboostingregressor__min_samples_leaf": [1, 2, 3],
    "gradientboostingregressor__max_depth": [2, 3, 5, 9]
}

moid_gbr_grid = GridSearchCV(
    pipelines_regression["Gradient Boosting Regressor"],
    moid_gbr_param_grid,
    cv=5,
    scoring="r2",
    n_jobs = -1,
    return_train_score=True
)



In [15]:
#There are null values in y_train_moid, hence the next three lines, which drop these rows before we're able to fit moid_gbr_grid 
# y_train_na_idx = y_train_moid[y_train_moid.isna()].index.tolist()
# y_train_moid.drop(y_train_na_idx, axis=0, inplace=True)
# X_train_no_moid.drop(y_train_na_idx, axis=0, inplace=True)

# moid_gbr_grid.fit(X_train_no_moid, y_train_moid)
# moid_best_model = moid_gbr_grid.best_estimator_

# y_na_idx = y_test_moid[y_test_moid.isna()].index.tolist()
# y_test_moid.drop(y_na_idx, axis=0, inplace=True)
# X_test_no_moid.drop(y_na_idx, axis=0, inplace=True)

# y_pred_moid = moid_best_model.predict(X_test_no_moid)



# print(f"GBR ---- R2 score: {r2_score(y_pred_moid, y_test_moid)}")




In [23]:

y_train_na_idx = y_train_moid[y_train_moid.isna()].index.tolist()
y_train_moid.drop(y_train_na_idx, axis=0, inplace=True)
X_train_no_moid.drop(y_train_na_idx, axis=0, inplace=True)

moid_best_model = pipelines_regression['Random Forest Regressor']
moid_best_model.fit(X_train_no_moid, y_train_moid)
with open('../artifacts/moid_best_model.pkl', 'wb') as file:
    joblib.dump(moid_best_model['randomforestregressor'], file)

column_transformer_moid = moid_best_model['columntransformer']
column_transformer_moid.fit(X_train_no_moid)

with open('../artifacts/column_transformer_moid.pkl', 'wb') as file:
    joblib.dump(column_transformer_moid, file)

#Saving reference train sets for monitoring
joblib.dump(X_train_no_moid, '../artifacts/data/X_train_no_moid.pkl')
joblib.dump(y_train_moid, '../artifacts/data/y_train_moid.pkl')

#Saving test sets for simulation of new data during monitoring
joblib.dump(X_test_no_moid, '../artifacts/data/X_test_no_moid.pkl')
joblib.dump(y_test_moid, '../artifacts/data/y_test_moid.pkl')

['../artifacts/data/y_test_moid.pkl']

In [22]:
X_train_no_moid_stats = X_train_no_moid.describe()
joblib.dump(X_train_no_moid_stats, "../artifacts/data/X_train_no_moid_stats.pkl")


['../artifacts/data/X_train_no_moid_stats.pkl']

In [ ]:
import os
from google.cloud import storage
storage.blob._MAX_MULTIPART_SIZE = 5 * 1024* 1024


def upload_local_directory(bucket_name, local_folder_path, gcs_folder_path=None):
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    for root, dirs, files in os.walk(local_folder_path):
        for file in files:
            local_file_path = os.path.join(root, file)
            print(f"local_file_path: {local_file_path}")
            print(f"dirs: {dirs}")

            relative_path = os.path.relpath(local_file_path, local_folder_path)
            print(f"relative_path: {relative_path}")
            gcs_destination = os.path.join(gcs_folder_path, relative_path ).replace("\\", "/")
            print(f"gcs_destination: {gcs_destination}")

            blob = bucket.blob(gcs_destination)
            blob._chunk_size = 5 * 1024* 1024
            blob.upload_from_filename(local_file_path)
            print(f"Uploaded {file} to {gcs_destination}")



upload_local_directory('project-3e6b348d-e2ae-4a47-9af_cloudbuild', '../artifacts','artifacts/')
    

local_file_path: ../artifacts/best_model_abs_mag.pkl
dirs: []
relative_path: best_model_abs_mag.pkl
gcs_destination: artifacts/best_model_abs_mag.pkl
Uploaded best_model_abs_mag.pkl to artifacts/best_model_abs_mag.pkl
local_file_path: ../artifacts/column_transformer_abs_mag.pkl
dirs: []
relative_path: column_transformer_abs_mag.pkl
gcs_destination: artifacts/column_transformer_abs_mag.pkl
Uploaded column_transformer_abs_mag.pkl to artifacts/column_transformer_abs_mag.pkl
local_file_path: ../artifacts/best_model_columntransformer.pkl
dirs: []
relative_path: best_model_columntransformer.pkl
gcs_destination: artifacts/best_model_columntransformer.pkl
Uploaded best_model_columntransformer.pkl to artifacts/best_model_columntransformer.pkl
local_file_path: ../artifacts/best_model.pkl
dirs: []
relative_path: best_model.pkl
gcs_destination: artifacts/best_model.pkl
Uploaded best_model.pkl to artifacts/best_model.pkl
local_file_path: ../artifacts/moid_bbest_model.pkl
dirs: []
relative_path: moi